# Lone-Wolf-Howler — guarded retraining

This notebook checks the source and output-suppression tests, stages only the approved JSONL files, and runs the recovery-aware training pipeline. GPU and Internet must be enabled. Add a Kaggle Secret named `HF_TOKEN` with a Hugging Face write token. The notebook reads it through Kaggle's secrets API and never embeds it in notebook source.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

WORK = Path('/kaggle/working')
REPO = WORK / 'Shadow-ronin-master'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/yalangisexter-ux/Shadow-ronin-master.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)

# Require the Kaggle Secret before spending GPU time; never inject the token into notebook source.
try:
    from kaggle_secrets import UserSecretsClient
    secrets_client = UserSecretsClient()
    hf_token = secrets_client.get_secret('HF_TOKEN')
except Exception as exc:
    raise RuntimeError(
        'Could not read the Kaggle Secret HF_TOKEN. Add it in Notebook > Add-ons > Secrets.'
    ) from exc

if not hf_token:
    raise RuntimeError('Kaggle Secret HF_TOKEN is empty.')
os.environ['HF_TOKEN'] = hf_token

# Publish scripts to Hugging Face before spending GPU time; HF_TOKEN comes from Kaggle Secrets.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub>=0.30.0'], check=True)
from huggingface_hub import HfApi, create_repo
hf_repo_id = 'aryansexter/Lone-Wolf-Howler'
try:
    hf_repo_id = secrets_client.get_secret('HF_REPO') or hf_repo_id
except Exception:
    pass  # HF_REPO is optional.
hf_repo_id = hf_repo_id.strip()
os.environ['HF_REPO'] = hf_repo_id
try:
    model_id_secret = secrets_client.get_secret('MODEL_ID')
    if model_id_secret:
        os.environ['MODEL_ID'] = model_id_secret.strip()
except Exception:
    pass  # MODEL_ID is optional.
api = HfApi(token=hf_token)
create_repo(repo_id=hf_repo_id, repo_type='model', private=False, exist_ok=True, token=hf_token)
publish_map = {
    'Lone-Wolf-Howler-End-to-End-Recovery-First-Fixed.py': 'scripts/Lone-Wolf-Howler-End-to-End-Recovery-First-Fixed.py',
    'Lone-Wolf-Howler-End-to-End-Recovery-First-Test.py': 'scripts/Lone-Wolf-Howler-End-to-End-Recovery-First-Test.py',
    'lone_wolf_inference.py': 'scripts/lone_wolf_inference.py',
    'tests/test_lone_wolf_inference.py': 'scripts/tests/test_lone_wolf_inference.py',
    'kaggle/lone-wolf-howler-retrain/Lone-Wolf-Howler-Retrain.ipynb': 'scripts/Lone-Wolf-Howler-Retrain.ipynb',
}
for source_rel, target_path in publish_map.items():
    source_path = REPO / source_rel
    if not source_path.is_file():
        raise FileNotFoundError(f'Required Hugging Face upload source is missing: {source_rel}')
    api.upload_file(path_or_fileobj=str(source_path), path_in_repo=target_path, repo_id=hf_repo_id, repo_type='model', commit_message=f'Sync Lone-Wolf-Howler script: {source_path.name}', token=hf_token)
    print(f'[hf-sync] uploaded {source_rel} -> {target_path}')
print(f'[hf-sync] completed script sync to {hf_repo_id}')

# Stage only approved datasets from repository files or attached Kaggle inputs.
# Deliberately exclude Dark_web.jsonl and Telugu_uncensored.jsonl.
DATA = WORK / 'lone_wolf_training_data'
DATA.mkdir(parents=True, exist_ok=True)
approved = ('combined_wolf_howler_dataset.jsonl', 'combined_clean_dataset_augmented.jsonl', 'combined_clean_dataset.jsonl')
found = []
search_roots = [REPO, Path('/kaggle/input')]
for name in approved:
    candidates = []
    for root in search_roots:
        if root.exists():
            candidates.extend(root.rglob(name))
    if candidates:
        src = candidates[0]
        dst = DATA / name
        if src.resolve() != dst.resolve():
            shutil.copy2(src, dst)
        found.append(name)
if not found:
    raise FileNotFoundError(
        'No approved training data found. Attach a Kaggle dataset containing '
        'combined_clean_dataset_augmented.jsonl and/or combined_clean_dataset.jsonl.'
    )
print(f"[dataset] staged approved files: {found}")

os.environ['TRAINING_DATA_DIR'] = str(DATA)
os.environ['FORCE_SCRATCH_TRAINING'] = '0'  # Prefer validated HF recovery; train only when no usable FP16 exists.
os.environ['ENABLE_GRPO'] = '0'
os.environ['DISABLE_RAG'] = '1'  # RAG remains disabled for this run.
os.environ.setdefault('MODEL_ID', 'Qwen/Qwen3-1.7B')

# Syntax and output-regression checks before training.
for name in ('Lone-Wolf-Howler-End-to-End-Recovery-First-Fixed.py',
             'Lone-Wolf-Howler-End-to-End-Recovery-First-Test.py',
             'Shadow-Ronin-4B-End-to-End-Recovery-First-Master.py',
             'lone_wolf_inference.py'):
    subprocess.run([sys.executable, '-m', 'py_compile', str(REPO / name)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(REPO / 'tests'), '-v'], check=True)

# Ensure bitsandbytes & accelerate are installed in this environment
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes>=0.43.0', 'accelerate', 'peft', 'gguf'], check=False)

# Run the fixed pipeline with automatic restart on environment repair (code 75)
max_attempts = 3
pipeline_completed = False
for attempt in range(1, max_attempts + 1):
    print(f"[pipeline] Starting pipeline execution (attempt {attempt}/{max_attempts})...")
    proc = subprocess.run(
        [sys.executable, str(REPO / 'Lone-Wolf-Howler-End-to-End-Recovery-First-Fixed.py')],
        cwd=str(REPO),
        env=os.environ.copy()
    )
    if proc.returncode == 75:
        raise RuntimeError(
            "[pipeline] bitsandbytes repaired the environment and explicitly requested a kernel restart (exit 75). "
            "A child-process retry cannot restart the Kaggle notebook kernel and may repeat the same failure. "
            "Restart the Kaggle session/kernel, then rerun this notebook from the top; staged datasets and HF artifacts "
            "are preserved according to the pipeline's recovery logic."
        )
    elif proc.returncode != 0:
        raise RuntimeError(f"Pipeline failed with exit code {proc.returncode}")
    else:
        print("[pipeline] Process exited with code 0; checking for a verified GGUF artifact now.", flush=True)
        pipeline_completed = True
        break

if not pipeline_completed:
    raise RuntimeError(f"Pipeline did not complete after {max_attempts} attempts; restart was repeatedly requested (exit code 75). Review the bitsandbytes preflight logs.")

# Finish with a validated GGUF in /kaggle/working/gguf, even when recovery reused an existing HF release.
working_gguf_dir = WORK / 'gguf'
local_ggufs = []
def verify_gguf_artifact(artifact):
    size = artifact.stat().st_size
    with artifact.open('rb') as fh:
        magic = fh.read(4)
    if size < 1024 * 1024 or magic != b'GGUF':
        raise RuntimeError(f'Invalid GGUF output: {artifact} ({size} bytes, magic={magic!r})')
    return size
if working_gguf_dir.is_dir():
    for artifact in sorted(working_gguf_dir.glob('*.gguf')):
        size = verify_gguf_artifact(artifact)
        local_ggufs.append((artifact, size))
        print(f'[gguf-output] verified {artifact} ({size} bytes)')
if not local_ggufs:
    remote_info = api.repo_info(repo_id=hf_repo_id, repo_type='model', files_metadata=True)
    remote_ggufs = [
        item for item in (getattr(remote_info, 'siblings', []) or [])
        if getattr(item, 'rfilename', '').startswith('gguf/')
        and getattr(item, 'rfilename', '').endswith('.gguf')
        and (getattr(item, 'size', None) is None or getattr(item, 'size', 0) > 0)
    ]
    if not remote_ggufs:
        raise RuntimeError('Pipeline exited successfully but no non-empty GGUF exists in /kaggle/working/gguf or on Hugging Face.')
    from huggingface_hub import hf_hub_download
    working_gguf_dir.mkdir(parents=True, exist_ok=True)
    download_errors = []
    for item in sorted(remote_ggufs, key=lambda entry: (getattr(entry, 'size', 0) or 0, entry.rfilename), reverse=True):
        try:
            cached_path = Path(hf_hub_download(repo_id=hf_repo_id, filename=item.rfilename, repo_type='model', token=hf_token))
            size = verify_gguf_artifact(cached_path)
            retained_path = working_gguf_dir / Path(item.rfilename).name
            if cached_path.resolve() != retained_path.resolve():
                shutil.copy2(cached_path, retained_path)
            size = verify_gguf_artifact(retained_path)
            local_ggufs.append((retained_path, size))
            print(f'[gguf-output] downloaded, verified, and retained {retained_path} ({size} bytes)')
            break
        except Exception as exc:
            download_errors.append(f'{item.rfilename}: {type(exc).__name__}: {exc}')
            print(f'[gguf-output] could not verify {item.rfilename}: {type(exc).__name__}: {exc}')
    if not local_ggufs:
        details = '; '.join(download_errors) or 'no downloadable GGUF candidates'
        raise RuntimeError(f'Pipeline exited successfully but no valid GGUF could be verified and retained; {details}')
print(f'[gguf-output] final retained GGUF count: {len(local_ggufs)}')
